In [17]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [18]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [19]:
TARGET_LOCATION = "location_name"
TARGET_COUNTRY  = "country"

In [20]:
target_col = TARGET_COUNTRY

# 1. Data Preparation

In [21]:
df = pd.read_csv(r"D:\My Folder\Dataset\GlobalWeatherRepository.csv")

features = [
    'temperature_celsius', 'wind_kph', 'wind_degree', 'pressure_mb', 
    'precip_mm', 'humidity', 'cloud', 'visibility_km', 'uv_index', 'gust_kph'
]


In [22]:
df = df.dropna(subset=features + [target_col])
X = df[features].values

In [23]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[target_col])
num_classes = len(label_encoder.classes_)
print(f"Number of unique classes to predict: {num_classes}")

Number of unique classes to predict: 211


In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, shuffle=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. PyTorch Dataset & DataLoader

In [25]:
class WeatherLocationDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [26]:
train_dataset = WeatherLocationDataset(X_train_scaled, y_train)
test_dataset = WeatherLocationDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 3. Model Definition

In [27]:
class LocationClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LocationClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim) 
        )
        
    def forward(self, x):
        return self.network(x)

In [28]:
input_dimension = len(features)
model = LocationClassifier(input_dim=input_dimension, output_dim=num_classes).to(device)

# 4. Training Loop & Optimisation

In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for batch_X, batch_y in train_loader:
        
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device).long()
        
        optimiser.zero_grad()
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        predicted = torch.argmax(outputs, dim=1)
        total_samples += batch_y.size(0)
        correct_predictions += (predicted == batch_y).sum().item()
        
        loss.backward()
        optimiser.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct_predictions / total_samples
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

RuntimeError: The size of tensor a (211) must match the size of tensor b (64) at non-singleton dimension 1

# 5. Save the Model

In [ ]:
model_path = "location_classifier.pth"
torch.save(model.state_dict(), model_path)
print(f"Model successfully saved to {model_path}")